In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import json
from pathlib import Path

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from utils.tools import aggregate_results

In [32]:
def get_suite(row):

    n_demos = row["number of demonstrations"]
    type_demos = row["type of demonstrations"][0:3]
    instr = "impl" if row["use instructions"] == "no" else "expl"

    return f"{n_demos}-{type_demos}-{instr}"

def capitalize(s):
    return s[0].upper() + s[1:]

def order_suites(strings):
    return sorted(strings, key=lambda s: (
        s.split('-')[2],  # implicit-excplict
        s.split('-')[0],  # num demos
        s.split('-')[1],  # demo type
    ))

def order_models(strings):
    return sorted(strings, key=lambda s: (
        int(s.split('-')[1][:-1]),  # Param count, B dropped
        #s.split('-')[0],  # Model name
    ), reverse=True)

def bold_extreme_values(s, by_model=True):
    # Bold max for mean

    if not by_model:
        is_max = s == s.max()
        return ['font-weight: bold' if v else '' for v in is_max]
    
    font_array = []

    model_level = s.index.names.index('model')
    models = s.index.get_level_values(model_level)
    models = pd.Series(list(models)).unique()

    idx = pd.IndexSlice
    
    for model in models:   
        values_by_model = s.loc[idx[model]]
        is_max = values_by_model == values_by_model.max()
        font_array += ['font-weight: bold' if v else '' for v in is_max]

    return font_array

In [38]:
res = pd.read_csv("./data/metrics.csv", sep=";")
#res = res.set_index(['model', 'number of demonstrations', 'type of demonstrations', 'use instructions'])
#res = res.sort_values(by=["use instructions", "number of demonstrations", "type of demonstrations"])

res["suite"] = res.apply(get_suite, axis=1)
res = res.set_index(["model", "suite"])

In [49]:
std_cols = [c for c in res.columns if "std" in c and "total" not in c]
std_df = res[std_cols].copy()

parts = std_df.columns.str.split(' ')

std_df.columns = pd.MultiIndex.from_tuples(
    [(p[0], p[1]) for p in parts],
    names=["label", "metric"]
)

pd.options.display.max_columns = None
pd.options.display.max_rows = None


#std_df


print(std_df.style.apply(bold_extreme_values).to_latex(convert_css=True))

\begin{tabular}{llrrrrrrrrrrrr}
 & label & \multicolumn{4}{r}{theme} & \multicolumn{4}{r}{topic} & \multicolumn{4}{r}{concept} \\
 & metric & accuracy & precision & recall & f1 & accuracy & precision & recall & f1 & accuracy & precision & recall & f1 \\
model & suite &  &  &  &  &  &  &  &  &  &  &  &  \\
\multirow[c]{11}{*}{Llama-8B} & 0-non-expl & 0.005050 & 0.003658 & 0.010354 & 0.003427 & 0.006255 & 0.005263 & 0.007231 & 0.007386 & 0.008627 & 0.006507 & 0.006995 & 0.014284 \\
 & 1-neg-impl & 0.050409 & 0.040178 & 0.096368 & \bfseries 0.144414 & 0.045867 & 0.043157 & 0.093303 & \bfseries 0.135791 & 0.044579 & 0.169873 & \bfseries 0.210885 & 0.126975 \\
 & 1-neg-expl & \bfseries 0.060690 & 0.051939 & 0.025177 & 0.110451 & 0.035447 & 0.020224 & 0.004622 & 0.115696 & 0.049023 & 0.037475 & 0.017312 & \bfseries 0.130548 \\
 & 1-pos-impl & 0.038072 & 0.017711 & 0.077596 & 0.035564 & 0.038017 & 0.000000 & 0.060607 & 0.038979 & 0.049261 & \bfseries 0.174165 & 0.155696 & 0.051680 \\
 & 1-pos